# NYC Taxi Demand & Revenue Analytics
This project aim is analyzes january 2025 NYC Yellow taxi records ti identify which pickup zones and time periods had the most recorded trips and highest fare revenue. 

In [15]:
from pathlib import Path
import pandas as pd 
# Locating the data file
raw_data_path = Path.home()/"Desktop"/"NYC_Taxi_Analytics"/"raw"
# loading the source table
trips = pd.read_parquet(
    raw_data_path/"yellow_tripdata_2025-01.parquet",
    engine ="pyarrow",
)
zones =pd.read_csv(raw_data_path/"taxi_zone_lookup.csv")
# checking dimensions of each path
print("trips records:",trips.shape)
print("zone records:",zones.shape)

trips records: (3475226, 20)
zone records: (265, 4)


In [17]:
# Preview the fields used to analyze trip demand and fare revenue
analysis_columns = [
    "tpep_pickup_datetime",
    "PULocationID",
    "fare_amount",
]
trips[analysis_columns].head()

,tpep_pickup_datetime,PULocationID,fare_amount
0,2025-01-01 00:18:38,229,10.0
1,2025-01-01 00:32:40,236,5.1
2,2025-01-01 00:44:04,141,5.1
3,2025-01-01 00:14:27,244,7.2
4,2025-01-01 00:21:34,244,5.8


In [19]:
# checking the data types and missing values in the analysis fields.
selected_fields = trips[analysis_columns]
print("data types")
print(selected_fields.dtypes)
print("\nmissing values")
print(selected_fields.isna().sum())

data types
tpep_pickup_datetime    datetime64[us]
PULocationID                     int32
fare_amount                    float64
dtype: object

missing values
tpep_pickup_datetime    0
PULocationID            0
fare_amount             0
dtype: int64


In [22]:
# checking the pickup dte range and count records ouside january in 2025
pickup_time = trips["tpep_pickup_datetime"]
print("Earlist pickup time:",pickup_time.min())
print("Letest pickup time:",pickup_time.max())
outside_january = (pickup_time < "2025-01-01") | (pickup_time >= "2025-02-01")
print("pickups outside january:",outside_january.sum())


Earlist pickup time: 2024-12-31 20:47:55
Letest pickup time: 2025-02-01 00:00:44
pickups outside january: 22


In [23]:
# Keeping only trips picked up january 2025
january_trips = trips[~outside_january]
print(f"january trip records: {len(january_trips):,}")

january trip records: 3,475,204


In [24]:
# counting zeo and negative fares amoung january pickups
fares = january_trips["fare_amount"]
print("zero fare records:",(fares == 0).sum())
print("negative fare records:",(fares < 0).sum())

zero fare records: 1398
negative fare records: 144118


In [25]:
# checking negative - fare records payment type 
negative_fares = january_trips.loc[
    january_trips["fare_amount"]<0,
    "payment_type",
]
print(negative_fares.value_counts(dropna = False))

payment_type
0    84822
4    37405
2    13802
3     8075
1       14
Name: count, dtype: int64


In [26]:
# comparing january fare total with and without negative fares 
fares = january_trips["fare_amount"]
print(f"Total including negative fares: ${fares.sum():,.2f}")
print(f"Total from positive fares: ${fares[fares>0].sum():,.2f}")
print(f"Sum of negative fares: ${fares[fares<0].sum():,.2f}")

Total including negative fares: $59,362,716.98
Total from positive fares: $61,038,943.39
Sum of negative fares: $-1,676,226.41


In [27]:
# count january records whose pickup ID is absent from the zone lookuos
pickup_has_match = january_trips["PULocationID"].isin(zones["LocationID"])
print("pickup location eithout a lookup match:",(~pickup_has_match).sum())

pickup location eithout a lookup match: 0


In [30]:
# checking any repeated ID in the zone lookup
duplicate_zone_ids = zones["LocationID"].duplicated()
print("Repeated zone IDs:",duplicate_zone_ids.sum())

Repeated zone IDs: 0


In [35]:
# for analysis merging the pickup zone and borough names with the fields
pickup_trips = january_trips[analysis_columns].merge(
    zones[["LocationID","Zone","Borough"]],
    how = "left",
    left_on = "PULocationID",
    right_on = "LocationID",
    validate = "many_to_one",
)
print(f"Rows after join:{len(pickup_trips):,}")
pickup_trips[["PULocationID","Zone","Borough"]].head()



Rows after join:3,475,204


,PULocationID,Zone,Borough
0,229,Sutton Place/Turtle Bay North,Manhattan
1,236,Upper East Side North,Manhattan
2,141,Lenox Hill West,Manhattan
3,244,Washington Heights South,Manhattan
4,244,Washington Heights South,Manhattan


In [36]:
# which pickup zones  has most recorded trips?
zone_summary = pickup_trips.groupby(
    ["PULocationID","Zone","Borough"],
    as_index = False,
    dropna = False,
).agg(
    trip_count =("fare_amount","size"),
    net_recorded_fare =("fare_amount","sum"),
)
zone_summary.sort_values("trip_count",ascending = False).head()



,PULocationID,Zone,Borough,trip_count,net_recorded_fare
156,161,Midtown Center,Manhattan,169977,2428899.69
232,237,Upper East Side South,Manhattan,163703,1921624.62
231,236,Upper East Side North,Manhattan,155647,1921094.88
127,132,JFK Airport,Queens,146135,8141144.50
225,230,Times Sq/Theatre District,Manhattan,125829,2045020.28


In [37]:
# pickup zones by net recorded fares
top_fare_zones = zone_summary.sort_values(
    "net_recorded_fare",
    ascending = False,
).head()
top_fare_zones[["Zone","Borough","trip_count","net_recorded_fare"]]



,Zone,Borough,trip_count,net_recorded_fare
127,JFK Airport,Queens,146135,8141144.50
133,LaGuardia Airport,Queens,89657,4458017.72
156,Midtown Center,Manhattan,169977,2428899.69
225,Times Sq/Theatre District,Manhattan,125829,2045020.28
232,Upper East Side South,Manhattan,163703,1921624.62


In [38]:
# gather hours of the day from the pickup timestamps
pickup_trips["pickup_hour"] = pickup_trips["tpep_pickup_datetime"].dt.hour
pickup_trips[["Zone","tpep_pickup_datetime","pickup_hour"]].head()


,Zone,tpep_pickup_datetime,pickup_hour
0,Sutton Place/Turtle Bay North,2025-01-01 00:18:38,0
1,Upper East Side North,2025-01-01 00:32:40,0
2,Lenox Hill West,2025-01-01 00:44:04,0
3,Washington Heights South,2025-01-01 00:14:27,0
4,Washington Heights South,2025-01-01 00:21:34,0


In [39]:
# counting record trips and sum fares by pickup hours and zones 
zone_hour_summary = pickup_trips.groupby(
    ["PULocationID","Zone","Borough","pickup_hour"],
    as_index = False,
    dropna = False
).agg(
    trip_count =("fare_amount","size"),
    net_recorded_fare = ("fare_amount","sum"),
)
zone_hour_summary.sort_values("trip_count",ascending = False).head()


,PULocationID,Zone,Borough,pickup_hour,trip_count,net_recorded_fare
3381,161,Midtown Center,Manhattan,18,17372,231770.13
3380,161,Midtown Center,Manhattan,17,16465,234962.31
3382,161,Midtown Center,Manhattan,19,13961,180972.76
4872,230,Times Sq/Theatre District,Manhattan,21,13579,182004.77
5010,236,Upper East Side North,Manhattan,15,13565,170315.66


In [40]:
# combination of zone-and-hour with highest records
zone_hour_summary.sort_values(
    "net_recorded_fare",
    ascending = False,
).head()

,PULocationID,Zone,Borough,pickup_hour,trip_count,net_recorded_fare
2848,138,LaGuardia Airport,Queens,12,5351,1082978.45
2708,132,JFK Airport,Queens,16,11596,701096.15
2707,132,JFK Airport,Queens,15,10357,615459.35
2711,132,JFK Airport,Queens,19,10247,582180.86
2712,132,JFK Airport,Queens,20,10453,577856.92


In [41]:
# Inspect the five largest fares for LaGuardia airpot pickups from noon to 1 pm
laguardia_noon = pickup_trips.loc[
    (pickup_trips["PULocationID"] == 138)
    &(pickup_trips["pickup_hour"] == 12),
    ["tpep_pickup_datetime","fare_amount"],
]
laguardia_noon.nlargest(5,"fare_amount")

,tpep_pickup_datetime,fare_amount
1780894,2025-01-20 12:07:18,863372.12
1401900,2025-01-16 12:34:46,220.00
99518,2025-01-02 12:55:48,200.00
2389008,2025-01-26 12:46:36,198.30
1193643,2025-01-14 12:00:30,195.50


In [45]:
# Inspect theoriginal recordbehind the unusually large lagurdia fare
laguardia_outlier = january_trips.loc[
    (january_trips["PULocationID"] == 138)
    &(january_trips["tpep_pickup_datetime"] == "2025-01-20 12:07:18")
    &(january_trips["fare_amount"] == 863372.12),
    ["trip_distance","RatecodeID","payment_type","fare_amount","total_amount"],
]
laguardia_outlier

,trip_distance,RatecodeID,payment_type,fare_amount,total_amount
1780915,1.6,1.0,4,863372.12,863380.37


In [47]:
# Inspecting the largest positive and negative fares accorss the january 
fare_check_columns = [
    "PULocationID","trip_distance","payment_type","fare_amount"
]
display(january_trips[fare_check_columns].nlargest(5,"fare_amount"))
display(january_trips[fare_check_columns].nsmallest(5,"fare_amount"))

,PULocationID,trip_distance,payment_type,fare_amount
1780915,138,1.60,4,863372.12
1404958,161,255.33,2,2450.90
1832645,247,188.88,2,1309.20
2358832,142,0.00,2,950.00
2358836,226,0.00,2,950.00


,PULocationID,trip_distance,payment_type,fare_amount
562142,226,0.10,3,-900.0
1760994,265,0.00,4,-850.0
74027,132,132.27,4,-826.2
38932,265,0.00,4,-700.0
38936,265,0.00,4,-700.0


In [48]:
# Flag the specific laguardia fare that distorted the houly ranking.
pickup_trips["fare_outlier_flag"] = (
    (pickup_trips["PULocationID"] == 138)
    &(pickup_trips["tpep_pickup_datetime"] == "2025-01-20 12:07:18")
    &(pickup_trips["fare_amount"] == 863372.12)
)
# we are separete the columns for adjested analysis.
pickup_trips["fare_for_analysis"] = pickup_trips["fare_amount"].mask(
    pickup_trips["fare_outlier_flag"]
)
print("flagged fare records:",pickup_trips["fare_outlier_flag"].sum())
print(f"adjusted net recordedfares: ${pickup_trips['fare_for_analysis'].sum():,.2f}")

flagged fare records: 1
adjusted net recordedfares: $58,499,344.86


In [51]:
# comparing original and adjusted fares for each pickup zone and hour.
adjusted_zone_hour_summary= pickup_trips.groupby(
    ["PULocationID","Zone","Borough","pickup_hour"],
    as_index = False,
    dropna = False,
).agg(
    trip_count = ("fare_amount","size"),
    net_recorded_fare = ("fare_amount","sum"),
    adjusted_net_recorded_fare = ("fare_for_analysis","sum")
)
adjusted_zone_hour_summary.nlargest(5,"adjusted_net_recorded_fare")

,PULocationID,Zone,Borough,pickup_hour,trip_count,net_recorded_fare,adjusted_net_recorded_fare
2708,132,JFK Airport,Queens,16,11596,701096.15,701096.15
2707,132,JFK Airport,Queens,15,10357,615459.35,615459.35
2711,132,JFK Airport,Queens,19,10247,582180.86,582180.86
2712,132,JFK Airport,Queens,20,10453,577856.92,577856.92
2709,132,JFK Airport,Queens,17,9763,575359.62,575359.62


In [52]:
# add up the metrices acress every pickup zone and hour.
summary = adjusted_zone_hour_summary
print(f"recorded pickups: {summary['trip_count'].sum():,}")
print(F"Original net fares: ${summary['net_recorded_fare'].sum():,.2f}")
print(f"Adujusted net records: ${summary['adjusted_net_recorded_fare'].sum():,.2f}")

recorded pickups: 3,475,204
Original net fares: $59,362,716.98
Adujusted net records: $58,499,344.86


In [55]:
# Export the processed data.
processed_dir = raw_data_path.parent/"data"/"processed"
processed_dir.mkdir(parents = True,exist_ok = True)
# we are selecting the trip fields needed for the SQL analysis.
export_columns = [
    "tpep_pickup_datetime",
    "PULocationID",
    "fare_amount",
    "fare_outlier_flag",
    "fare_for_analysis",
]
# write one row per recorded january pickup.
export_path = processed_dir/"january_2025_pickups.csv"
pickup_trips[export_columns].to_csv(export_path,index =False)
print(f"Exported{len(pickup_trips):,} rows to {export_path}")

Exported3,475,204 rows to /Users/naveenchunduri/Desktop/NYC_Taxi_Analytics/data/processed/january_2025_pickups.csv
